# Random effects: the union wage premium

**Maths for Data Science III · Panel data**

This short notebook fits one model, **random effects**, to Wooldridge's `wagepan` panel and shows how to read its output. It asks: how much more do union members earn, once we account for the fact that each person is observed several times?

For the full comparison with pooled OLS and fixed effects, and the tests for choosing between them, see `wagepan_fixed_vs_random_effects.ipynb`.

## 0. Setup

In [1]:
!pip install wooldridge linearmodels statsmodels

import numpy as np
import pandas as pd
import statsmodels.api as sm
import wooldridge
from linearmodels.panel import RandomEffects

pd.set_option("display.precision", 4)
pd.set_option("display.width", 120)

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 10.0 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


## 1. The data

`wagepan` follows **545 young American men every year from 1980 to 1987** (4,360 rows).

| Variable | Meaning |
|---|---|
| `nr`, `year` | person identifier, year |
| `lwage` | log of hourly wage (the outcome) |
| `union` | 1 if wage set by a collective bargaining agreement |
| `married` | 1 if married |
| `educ` | years of schooling (does not change over the period) |
| `black`, `hisp` | 1 if Black, 1 if Hispanic |
| `exper`, `expersq` | years of experience and its square |
| `d81` … `d87` | year dummies (1980 is the reference year) |

Because the outcome is in logs, a coefficient $b$ means roughly a $100 \times (e^{b}-1)\%$ difference in wages.

`linearmodels` needs a two-level index: the unit (`nr`) first, then time (`year`).

In [2]:
df = wooldridge.data("wagepan").set_index(["nr", "year"])
print(df.shape)
df[["lwage", "union", "married", "educ", "black", "hisp", "exper"]].head()

(4360, 42)


lwage  union  married  educ  black  hisp  exper
nr year                                                  
13 1980  1.1975      0        0    14      0     0      1
   1981  1.8531      1        0    14      0     0      2
   1982  1.3445      0        0    14      0     0      3
   1983  1.4332      0        0    14      0     0      4
   1984  1.5681      0        0    14      0     0      5

## 2. The random effects model

$$
\text{lwage}_{it} = \beta_0 + \beta_1\,\text{union}_{it} + \beta_2\,\text{married}_{it} + \dots + a_i + u_{it}
$$

- $a_i$ is the **individual effect**: a stable, unmeasured wage level for person $i$ (ability, motivation, background).
- $u_{it}$ is the ordinary year-to-year noise.

**The key assumption:** random effects treats $a_i$ as random noise that is **unrelated to the explanatory variables** in every year. In this example, that means union members are no more (or less) likely than others to have a high hidden wage level.

**How it works:** because each person's $a_i$ appears in all eight years, their errors are correlated. Random effects accounts for this by subtracting a fraction $\theta$ of each person's average from every year:

$$
\theta = 1 - \sqrt{\frac{\sigma_u^2}{\sigma_u^2 + T\,\sigma_a^2}}
$$

where $\sigma_a^2$ is the variance of the individual effects, $\sigma_u^2$ the variance of the noise and $T$ the number of years. When $\theta = 0$ the model is pooled OLS; as $\theta \to 1$ it approaches fixed effects. Because only part of each average is removed, variables that never change for a person, such as education and race, stay in the model.

In the clinical literature this same model is called a **random-intercept** (multilevel or mixed) model.

## 3. Fit the model

We include the time-invariant variables, experience, and year dummies. Standard errors are **clustered by person**, because each man contributes eight related observations.

In [3]:
year_dummies = ["d81", "d82", "d83", "d84", "d85", "d86", "d87"]
X = sm.add_constant(df[["educ", "black", "hisp", "exper", "expersq", "married", "union"] + year_dummies])

re = RandomEffects(df["lwage"], X).fit(cov_type="clustered", cluster_entity=True)
print(re.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                  lwage   R-squared:                        0.1806
Estimator:              RandomEffects   R-squared (Between):              0.1853
No. Observations:                4360   R-squared (Within):               0.1799
Date:                Thu, Oct 29 2026   R-squared (Overall):              0.1828
Time:                        09:40:32   Log-likelihood                   -1622.5
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      68.409
Entities:                         545   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                 F(14,4345)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             43.681
                            

## 4. How much of the variation is a stable person effect?

In [4]:
vd = re.variance_decomposition
sigma2_a, sigma2_u, T = vd["Effects"], vd["Residual"], 8
theta_by_hand = 1 - np.sqrt(sigma2_u / (sigma2_u + T * sigma2_a))

print(vd.round(4))
print(f"\ntheta from linearmodels: {re.theta.iloc[0, 0]:.3f}")
print(f"theta by hand:           {theta_by_hand:.3f}")

Effects                   0.1069
Residual                  0.1233
Percent due to Effects    0.4644
Name: Variance Decomposition, dtype: float64

theta from linearmodels: 0.645
theta by hand:           0.645


About **46%** of the unexplained variation in log wages is a stable person effect: some men are consistently higher earners than others. With 8 years per man, that gives $\theta \approx 0.64$, so random effects removes nearly two-thirds of each man's average. It sits between pooled OLS ($\theta = 0$) and fixed effects ($\theta = 1$).

## 5. Interpreting the coefficients

In [5]:
key = ["union", "married", "educ", "black", "hisp"]
ci = re.conf_int().loc[key]
pd.DataFrame({
    "coef": re.params[key],
    "SE (clustered)": re.std_errors[key],
    "p-value": re.pvalues[key],
    "% difference": 100 * (np.exp(re.params[key]) - 1),
    "95% CI low (%)": 100 * (np.exp(ci["lower"]) - 1),
    "95% CI high (%)": 100 * (np.exp(ci["upper"]) - 1),
}).round(3)

,coef,SE (clustered),p-value,% difference,95% CI low (%),95% CI high (%)
union,0.106,0.021,0.000,11.167,6.718,15.802
married,0.064,0.019,0.001,6.589,2.699,10.627
educ,0.092,0.011,0.000,9.623,7.256,12.043
black,-0.139,0.051,0.006,-13.010,-21.270,-3.884
hisp,0.022,0.040,0.586,2.199,-5.489,10.511


Holding the other variables fixed, the random-effects model estimates that:

- **Union coverage** is associated with about **11% higher wages** (95% CI about 7% to 16%).
- **Being married** is associated with about **7% higher wages**.
- **Each extra year of schooling** is associated with about **10% higher wages**.
- **Black men** earn about **13% less** than otherwise similar white men; the difference for **Hispanic men** is small and not statistically significant.

Random effects can estimate the schooling and race effects because it keeps the variables that never change for a person. A fixed-effects model could not.